In [1]:
!python -V

Python 3.14.4


In [2]:
import pandas as pd
import numpy as np
import pickle

In [3]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.metrics import root_mean_squared_error

In [4]:
import mlflow

# mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("nyc-taxi-experiment")

2026/05/08 11:22:01 INFO mlflow.tracking.fluent: Experiment with name 'nyc-taxi-experiment' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1778257321937, experiment_id='1', last_update_time=1778257321937, lifecycle_stage='active', name='nyc-taxi-experiment', tags={}, trace_location=None, workspace='default'>

In [9]:
# Try using fastparquet if you have it installed
# df = pd.read_parquet('./data/green_tripdata_2021-01.parquet', engine='fastparquet')
def read_dataframe(filename):

    df = pd.read_parquet(filename, engine="fastparquet")

    df['duration'] = df.lpep_dropoff_datetime - df.lpep_pickup_datetime
    df.duration = df.duration.apply(lambda td: td.total_seconds() / 60)

    df = df[(df.duration >= 1) & (df.duration <= 60)]

    categorical = ['PULocationID', 'DOLocationID']
    df[categorical] = df[categorical].astype(str)

    df["PU_DO"] = df["PULocationID"] + '_' + df["DOLocationID"]
    
    return df

In [10]:
df_train = read_dataframe('https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2021-01.parquet')
df_val = read_dataframe('https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2021-02.parquet')

In [11]:
categorical = ['PU_DO'] #'PULocationID', 'DOLocationID']
numerical = ['trip_distance']

dv = DictVectorizer()

train_dicts = df_train[categorical + numerical].to_dict(orient='records')
X_train = dv.fit_transform(train_dicts)

val_dicts = df_val[categorical + numerical].to_dict(orient='records')
X_val = dv.transform(val_dicts)

In [12]:

target = 'duration'
y_train = df_train[target].values
y_val = df_val[target].values

In [16]:
import xgboost as xgb 

In [19]:
from pathlib import Path

models_folder = Path('models')
models_folder.mkdir(exist_ok=True)

In [22]:
mlflow.xgboost.autolog(disable=True)

In [23]:
with mlflow.start_run():
    train = xgb.DMatrix(X_train, label=y_train)
    valid = xgb.DMatrix(X_val, label=y_val)

    best_params = {
        "max_depth": 20,
        "learning_rate": 0.666730261582146,
        "reg_alpha": 0.06229964677766508,
        "reg_lambda": 0.02599517340753995,
        "min_child_weight": 1.4071004781961702,
        "seed": 42
    }

    mlflow.log_params(best_params)

    booster = xgb.train(
        params = best_params,
        dtrain = train,
        num_boost_round = 100,
        evals = [(valid, "validation")],
        early_stopping_rounds = 5
    )

    y_pred = booster.predict(valid)
    rmse = root_mean_squared_error(y_val, y_pred)
    mlflow.log_metric("rmse", rmse)
    with open("models/preprocessor.b", "wb") as f_out:
        pickle.dump(dv, f_out)
    mlflow.log_artifact("models/preprocessor.b", artifact_path="preprocessor")
    mlflow.xgboost.log_model(booster, artifact_path = "models_mlflow")

[0]	validation-rmse:7.69164
[1]	validation-rmse:6.85656
[2]	validation-rmse:6.68397
[3]	validation-rmse:6.64277
[4]	validation-rmse:6.63128
[5]	validation-rmse:6.62306
[6]	validation-rmse:6.61656
[7]	validation-rmse:6.60618
[8]	validation-rmse:6.60010
[9]	validation-rmse:6.59891
[10]	validation-rmse:6.59252
[11]	validation-rmse:6.58741
[12]	validation-rmse:6.58384
[13]	validation-rmse:6.58004
[14]	validation-rmse:6.57691
[15]	validation-rmse:6.57066
[16]	validation-rmse:6.56502
[17]	validation-rmse:6.55772
[18]	validation-rmse:6.55468
[19]	validation-rmse:6.54978
[20]	validation-rmse:6.54608
[21]	validation-rmse:6.54084
[22]	validation-rmse:6.53772
[23]	validation-rmse:6.53244
[24]	validation-rmse:6.52987
[25]	validation-rmse:6.52680
[26]	validation-rmse:6.52565
[27]	validation-rmse:6.52118
[28]	validation-rmse:6.51778
[29]	validation-rmse:6.51449
[30]	validation-rmse:6.51198
[31]	validation-rmse:6.51025
[32]	validation-rmse:6.50360
[33]	validation-rmse:6.50226
[34]	validation-rmse:6.5

2026/05/08 11:45:59 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/08 11:46:07 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run abundant-perch-283 at: http://127.0.0.1:5000/#/experiments/1/runs/3ba74992c807430b9a85e69cb9df6b0a
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
